# W7D1: Haystack Pipeline Architecture

## Objectives
- Build a Haystack pipeline using FileConverter, DocumentStore, BM25Retriever, and Reader.
- Index 5 PDF documents.
- Test retrieval with 10 questions.
- Compare BM25 and DenseRetriever performance.

**Environment:** Python 3.14 virtual environment
**Framework:** Haystack 3.3.0

In [1]:
import haystack

print("Haystack version:", haystack.__version__)
print("Haystack import successful!")

Haystack version: 3.3.0
Haystack import successful!


In [2]:
from haystack import Pipeline
from haystack.document_stores.in_memory import InMemoryDocumentStore
from haystack.components.retrievers.in_memory import InMemoryBM25Retriever

print("Pipeline:", Pipeline)
print("DocumentStore:", InMemoryDocumentStore)
print("BM25 Retriever:", InMemoryBM25Retriever)
print("Haystack components loaded successfully!")

Pipeline: <class 'haystack.core.pipeline.pipeline.Pipeline'>
DocumentStore: <class 'haystack.document_stores.in_memory.document_store.InMemoryDocumentStore'>
BM25 Retriever: <class 'haystack.components.retrievers.in_memory.bm25_retriever.InMemoryBM25Retriever'>
Haystack components loaded successfully!


In [4]:
from haystack.components.converters import PyPDFToDocument

print("PDF converter:", PyPDFToDocument)
print("PDF converter loaded successfully!")

PDF converter: <class 'haystack.components.converters.pypdf.PyPDFToDocument'>
PDF converter loaded successfully!


In [6]:
from haystack_integrations.components.readers.transformers import TransformersExtractiveReader

print("Reader:", TransformersExtractiveReader)
print("Reader import successful!")

Reader: <class 'haystack_integrations.components.readers.transformers.extractive_reader.TransformersExtractiveReader'>
Reader import successful!


In [7]:
from haystack import Pipeline
from haystack.document_stores.in_memory import InMemoryDocumentStore
from haystack.components.retrievers.in_memory import InMemoryBM25Retriever

document_store = InMemoryDocumentStore()
retriever = InMemoryBM25Retriever(document_store=document_store)

print("DocumentStore created!")
print("BM25 Retriever created!")

DocumentStore created!
BM25 Retriever created!


In [8]:
from haystack_integrations.components.readers.transformers import TransformersExtractiveReader

reader = TransformersExtractiveReader(model="deepset/roberta-base-squad2")

print("Reader created!")
print("Model:", reader.model)

Reader created!
Model: None


In [9]:
from haystack import Document

test_docs = [
    Document(content="ROS 2 is a robotics middleware framework."),
    Document(content="LiDAR is a sensor used to measure distance."),
]

result = reader.run(
    query="What is ROS 2?",
    documents=test_docs,
    top_k=1
)

print("Reader test successful!")
print(result["answers"][0])

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

c:\Users\ughan\CynarisInternship\AIML_PROJECT\venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ughan\.cache\huggingface\hub\models--deepset--roberta-base-squad2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
c:\Users\ughan\CynarisInternship\AIML_PROJECT\venv\Lib\site-packages\huggingface_hub\file_

model.safetensors: reconstructing file:   0%|          |  0.00B /  496MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/79.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

Reader test successful!
ExtractedAnswer(query='What is ROS 2?', score=0.838811457157135, data='a robotics middleware framework', document=Document(id=e94428450e6f6058f709ceb778c6788bf2a01a6df41ae249ba4451f5c5e32b82, content: 'ROS 2 is a robotics middleware framework.'), context=None, document_offset=ExtractedAnswer.Span(start=9, end=40), context_offset=None, meta={})


In [10]:
from haystack.components.converters import PyPDFToDocument

converter = PyPDFToDocument()

result = converter.run(
    sources=["notebooks/robotics_reference.pdf"]
)

documents = result["documents"]

print("Documents extracted:", len(documents))
print("First document preview:")
print(documents[0].content[:500])

Could not read notebooks/robotics_reference.pdf. Skipping it. Error: [Errno 2] No such file or directory: 'notebooks\\robotics_reference.pdf'


Documents extracted: 0
First document preview:


IndexError: list index out of range

In [11]:
from pathlib import Path
from haystack.components.converters import PyPDFToDocument

pdf_path = Path(r"C:\Users\ughan\CynarisInternship\AIML_PROJECT\notebooks\robotics_reference.pdf")

converter = PyPDFToDocument()

result = converter.run(sources=[pdf_path])
documents = result["documents"]

print("Documents extracted:", len(documents))
print("First document preview:")
print(documents[0].content[:500])

Documents extracted: 1
First document preview:
Robotics Overview
Robotics combines mechanical engineering, electronics, control systems, and computer science.
Robots use sensors to perceive their surroundings and actuators to produce movement.
Sensors and Perception
LiDAR measures distance using laser light and helps robots build maps. Ultrasonic sensors
estimate distance using sound waves. Cameras provide image data for computer vision. An IMU
measures angular velocity and linear acceleration, while wheel encoders measure wheel rotation.
Lo


In [12]:
document_store.write_documents(documents)

print("Documents indexed:", document_store.count_documents())

Documents indexed: 1


In [13]:
from haystack import Pipeline

bm25_pipeline = Pipeline()

bm25_pipeline.add_component(
    instance=retriever,
    name="retriever"
)

bm25_pipeline.add_component(
    instance=reader,
    name="reader"
)

bm25_pipeline.connect(
    "retriever.documents",
    "reader.documents"
)

print("BM25 → Reader pipeline created successfully!")

BM25 → Reader pipeline created successfully!


In [14]:
question = "What does LiDAR measure?"

result = bm25_pipeline.run(
    data={
        "retriever": {
            "query": question,
            "top_k": 3
        },
        "reader": {
            "query": question,
            "top_k": 1
        }
    }
)

print("Question:", question)
print("Answer:", result["reader"]["answers"][0])

Question: What does LiDAR measure?
Answer: ExtractedAnswer(query='What does LiDAR measure?', score=0.7706772685050964, data='distance', document=Document(id=3183c23d037412f8304aae9dce944698625c84c8b73756fd0328e89f5d7ec450, content: 'Robotics Overview
Robotics combines mechanical engineering, electronics, control systems, and comput...', meta: {'file_path': 'robotics_reference.pdf'}, score: 0.7192051811294521), context=None, document_offset=ExtractedAnswer.Span(start=235, end=243), context_offset=None, meta={})


In [15]:
questions = [
    "What does LiDAR measure?",
    "What does an IMU measure?",
    "What do wheel encoders measure?",
    "What is the purpose of sensors in robots?",
    "What is SLAM used for?",
    "What is the purpose of a PID controller?",
    "What is ROS 2 used for?",
    "What do cameras provide for robots?",
    "What do ultrasonic sensors estimate?",
    "What do actuators produce?"
]

print("10-question evaluation set created!")
print("Number of questions:", len(questions))

10-question evaluation set created!
Number of questions: 10


In [ ]:
bm25_results = []

for i, question in enumerate(questions, start=1):
    result = bm25_pipeline.run(
        data={
            "retriever": {
                "query": question,
                "top_k": 3
            },
            "reader": {
                "query": question,
                "top_k": 1
            }
        }
    )

    answer = result["reader"]["answers"][0] if result["reader"]["answers"] else None

    bm25_results.append({
        "question": question,
        "answer": answer.data if answer else "No answer",
        "reader_score": answer.score if answer else 0,
        "retrieval_score": answer.document.score if answer and answer.document else 0
    })

    print(f"{i}. {question}")
    print(f"   Answer: {answer.data if answer else 'No answer'}")
    print(f"   Reader score: {answer.score:.3f}" if answer else "   Reader score: N/A")
    print()

1. What does LiDAR measure?
   Answer: distance
   Reader score: 0.771

2. What does an IMU measure?
   Answer: angular velocity and linear acceleration
   Reader score: 0.837

3. What do wheel encoders measure?
   Answer: wheel rotation
   Reader score: 0.816

4. What is the purpose of sensors in robots?
   Answer: to perceive their surroundings and actuators to produce movement
   Reader score: 0.765

5. What is SLAM used for?
   Answer: allows a robot to build a map while estimating its own position
   Reader score: 0.757

6. What is the purpose of a PID controller?
   Answer: to regulate a system
   Reader score: 0.748

7. What is ROS 2 used for?
   Answer: communication tools for robot
applications
   Reader score: 0.543

8. What do cameras provide for robots?
   Answer: image data for computer vision
   Reader score: 0.831

9. What do ultrasonic sensors estimate?
   Answer: distance
   Reader score: 0.758

10. What do actuators produce?
   Answer: movement
   Reader score: 0.747


In [17]:
import pandas as pd

bm25_df = pd.DataFrame(bm25_results)

display(bm25_df[[
    "question",
    "answer",
    "reader_score",
    "retrieval_score"
]])

,question,answer,reader_score,retrieval_score
0,What does LiDAR measure?,distance,0.770677,0.719205
1,What does an IMU measure?,angular velocity and linear acceleration,0.837095,1.078808
2,What do wheel encoders measure?,wheel rotation,0.816120,1.168708
3,What is the purpose of sensors in robots?,to perceive their surroundings and actuators t...,0.765216,2.031755
4,What is SLAM used for?,allows a robot to build a map while estimating...,0.756510,0.809106
5,What is the purpose of a PID controller?,to regulate a system,0.748273,2.329244
6,What is ROS 2 used for?,communication tools for robot\napplications,0.543348,0.899006
7,What do cameras provide for robots?,image data for computer vision,0.831476,1.672152
8,What do ultrasonic sensors estimate?,distance,0.758354,1.258609
9,What do actuators produce?,movement,0.747248,0.719205


In [18]:
from haystack.components.retrievers.in_memory import InMemoryEmbeddingRetriever

print("Dense retriever:", InMemoryEmbeddingRetriever)
print("Dense retriever import successful!")

Dense retriever: <class 'haystack.components.retrievers.in_memory.embedding_retriever.InMemoryEmbeddingRetriever'>
Dense retriever import successful!


In [19]:
from haystack_integrations.components.embedders.sentence_transformers import (
    SentenceTransformersDocumentEmbedder,
    SentenceTransformersTextEmbedder,
)

print("Document embedder:", SentenceTransformersDocumentEmbedder)
print("Text embedder:", SentenceTransformersTextEmbedder)
print("Sentence Transformers integration loaded successfully!")

Document embedder: <class 'haystack_integrations.components.embedders.sentence_transformers.sentence_transformers_document_embedder.SentenceTransformersDocumentEmbedder'>
Text embedder: <class 'haystack_integrations.components.embedders.sentence_transformers.sentence_transformers_text_embedder.SentenceTransformersTextEmbedder'>
Sentence Transformers integration loaded successfully!


In [20]:
document_embedder = SentenceTransformersDocumentEmbedder(
    model="sentence-transformers/all-MiniLM-L6-v2"
)

document_embedder.warm_up()

print("Document embedder ready!")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

c:\Users\ughan\CynarisInternship\AIML_PROJECT\venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ughan\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Document embedder ready!


In [21]:
embedded_documents = document_embedder.run(documents)

documents_with_embeddings = embedded_documents["documents"]

print("Documents embedded:", len(documents_with_embeddings))
print(
    "Embedding dimension:",
    len(documents_with_embeddings[0].embedding)
)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Documents embedded: 1
Embedding dimension: 384


In [22]:
document_store.write_documents(documents_with_embeddings)

print("Total documents in DocumentStore:", document_store.count_documents())


DuplicateDocumentError: ID '3183c23d037412f8304aae9dce944698625c84c8b73756fd0328e89f5d7ec450' already exists.

In [23]:
from haystack.document_stores.types import DuplicatePolicy

document_store.write_documents(
    documents_with_embeddings,
    policy=DuplicatePolicy.OVERWRITE
)

print("Documents in DocumentStore:", document_store.count_documents())
print("Embedded document stored successfully!")

Documents in DocumentStore: 1
Embedded document stored successfully!


In [24]:
dense_retriever = InMemoryEmbeddingRetriever(
    document_store=document_store
)

print("Dense Retriever created successfully!")

Dense Retriever created successfully!


In [25]:
dense_pipeline = Pipeline()

dense_pipeline.add_component(
    instance=dense_retriever,
    name="retriever"
)

dense_pipeline.add_component(
    instance=reader,
    name="reader"
)

dense_pipeline.connect(
    "retriever.documents",
    "reader.documents"
)

print("Dense Retriever → Reader pipeline created successfully!")

PipelineError: Component has already been added in another Pipeline. Components can't be shared between Pipelines. Create a new instance instead.

In [26]:
dense_reader = TransformersExtractiveReader(
    model="deepset/roberta-base-squad2"
)

dense_reader.warm_up()

print("Dense pipeline Reader created successfully!")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Dense pipeline Reader created successfully!


In [27]:
 dense_pipeline = Pipeline()

dense_pipeline.add_component(
    instance=dense_retriever,
    name="retriever"
)

dense_pipeline.add_component(
    instance=dense_reader,
    name="reader"
)

dense_pipeline.connect(
    "retriever.documents",
    "reader.documents"
)

print("Dense Retriever → Reader pipeline created successfully!")

IndentationError: unexpected indent (3054991025.py, line 1)

In [28]:
dense_pipeline = Pipeline()

dense_pipeline.add_component(
    instance=dense_retriever,
    name="retriever"
)

dense_pipeline.add_component(
    instance=dense_reader,
    name="reader"
)

dense_pipeline.connect(
    "retriever.documents",
    "reader.documents"
)

print("Dense Retriever → Reader pipeline created successfully!")

PipelineError: Component has already been added in another Pipeline. Components can't be shared between Pipelines. Create a new instance instead.

In [29]:
dense_pipeline.add_component(
    instance=dense_reader,
    name="reader"
)

dense_pipeline.connect(
    "retriever.documents",
    "reader.documents"
)

print("Dense Retriever → Reader pipeline created successfully!")

ValueError: Component named retriever not found in the pipeline.

In [30]:
from haystack.components.retrievers.in_memory import InMemoryEmbeddingRetriever
from haystack_integrations.components.readers.transformers import TransformersExtractiveReader

fresh_dense_retriever = InMemoryEmbeddingRetriever(
    document_store=document_store
)

fresh_dense_reader = TransformersExtractiveReader(
    model="deepset/roberta-base-squad2"
)

fresh_dense_reader.warm_up()

fresh_dense_pipeline = Pipeline()

fresh_dense_pipeline.add_component(
    instance=fresh_dense_retriever,
    name="retriever"
)

fresh_dense_pipeline.add_component(
    instance=fresh_dense_reader,
    name="reader"
)

fresh_dense_pipeline.connect(
    "retriever.documents",
    "reader.documents"
)

print("Fresh Dense Retriever → Reader pipeline created successfully!")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Fresh Dense Retriever → Reader pipeline created successfully!


In [31]:
dense_results = []

for i, question in enumerate(questions, start=1):
    query_embedding = SentenceTransformersTextEmbedder(
        model="sentence-transformers/all-MiniLM-L6-v2"
    )
    query_embedding.warm_up()

    query_result = query_embedding.run(text=question)

    result = fresh_dense_pipeline.run(
        data={
            "retriever": {
                "query_embedding": query_result["embedding"],
                "top_k": 3
            },
            "reader": {
                "query": question,
                "top_k": 1
            }
        }
    )

    answer = result["reader"]["answers"][0] if result["reader"]["answers"] else None

    dense_results.append({
        "question": question,
        "answer": answer.data if answer else "No answer",
        "reader_score": answer.score if answer else 0,
        "retrieval_score": answer.document.score if answer and answer.document else 0
    })

    print(f"{i}. {question}")
    print(f"   Answer: {answer.data if answer else 'No answer'}")
    print()

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

1. What does LiDAR measure?
   Answer: distance



Batches:   0%|          | 0/1 [00:00<?, ?it/s]

2. What does an IMU measure?
   Answer: angular velocity and linear acceleration



Batches:   0%|          | 0/1 [00:00<?, ?it/s]

...

8. What do cameras provide for robots?
   Answer: image data for computer vision



Batches:   0%|          | 0/1 [00:00<?, ?it/s]

9. What do ultrasonic sensors estimate?
   Answer: distance



Batches:   0%|          | 0/1 [00:00<?, ?it/s]

10. What do actuators produce?
   Answer: movement



In [32]:
import pandas as pd

comparison_df = pd.DataFrame({
    "Question": questions,
    "BM25 Answer": [r["answer"] for r in bm25_results],
    "BM25 Score": [r["reader_score"] for r in bm25_results],
    "Dense Answer": [r["answer"] for r in dense_results],
    "Dense Score": [r["reader_score"] for r in dense_results],
})

display(comparison_df)

,Question,BM25 Answer,BM25 Score,Dense Answer,Dense Score
0,What does LiDAR measure?,distance,0.770677,distance,0.770677
1,What does an IMU measure?,angular velocity and linear acceleration,0.837095,angular velocity and linear acceleration,0.837095
2,What do wheel encoders measure?,wheel rotation,0.816120,wheel rotation,0.816120
3,What is the purpose of sensors in robots?,to perceive their surroundings and actuators t...,0.765216,to perceive their surroundings and actuators t...,0.765216
4,What is SLAM used for?,allows a robot to build a map while estimating...,0.756510,allows a robot to build a map while estimating...,0.756510
5,What is the purpose of a PID controller?,to regulate a system,0.748273,to regulate a system,0.748273
6,What is ROS 2 used for?,communication tools for robot\napplications,0.543348,communication tools for robot\napplications,0.543348
7,What do cameras provide for robots?,image data for computer vision,0.831476,image data for computer vision,0.831476
8,What do ultrasonic sensors estimate?,distance,0.758354,distance,0.758354
9,What do actuators produce?,movement,0.747248,movement,0.747248


In [33]:
query_embedder = SentenceTransformersTextEmbedder(
    model="sentence-transformers/all-MiniLM-L6-v2"
)

query_embedder.warm_up()

print("Query embedder ready!")

Query embedder ready!


In [34]:
dense_results_clean = []

for i, question in enumerate(questions, start=1):
    query_result = query_embedder.run(text=question)

    result = fresh_dense_pipeline.run(
        data={
            "retriever": {
                "query_embedding": query_result["embedding"],
                "top_k": 3
            },
            "reader": {
                "query": question,
                "top_k": 1
            }
        }
    )

    answer = result["reader"]["answers"][0] if result["reader"]["answers"] else None

    dense_results_clean.append({
        "question": question,
        "answer": answer.data if answer else "No answer",
        "reader_score": answer.score if answer else 0,
        "retrieval_score": answer.document.score if answer and answer.document else 0
    })

print("Clean Dense evaluation completed!")
print("Questions tested:", len(dense_results_clean))

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

...

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Clean Dense evaluation completed!
Questions tested: 10


In [35]:
final_comparison = pd.DataFrame({
    "Question": questions,
    "BM25 Answer": [r["answer"] for r in bm25_results],
    "Dense Answer": [r["answer"] for r in dense_results_clean],
})

display(final_comparison)

,Question,BM25 Answer,Dense Answer
0,What does LiDAR measure?,distance,distance
1,What does an IMU measure?,angular velocity and linear acceleration,angular velocity and linear acceleration
2,What do wheel encoders measure?,wheel rotation,wheel rotation
3,What is the purpose of sensors in robots?,to perceive their surroundings and actuators t...,to perceive their surroundings and actuators t...
4,What is SLAM used for?,allows a robot to build a map while estimating...,allows a robot to build a map while estimating...
5,What is the purpose of a PID controller?,to regulate a system,to regulate a system
6,What is ROS 2 used for?,communication tools for robot\napplications,communication tools for robot\napplications
7,What do cameras provide for robots?,image data for computer vision,image data for computer vision
8,What do ultrasonic sensors estimate?,distance,distance
9,What do actuators produce?,movement,movement


## Evaluation & Conclusion

### BM25 Retrieval
- Tested 10 robotics questions.
- Manually correct answers: 10/10.
- Manual answer accuracy: 100%.

### Dense Retrieval
- Tested the same 10 robotics questions.
- Manually correct answers: 10/10.
- Manual answer accuracy: 100%.

### Comparison
- BM25 and Dense Retrieval produced the same answers for all 10 test questions.
- Both achieved 100% manual answer accuracy on this test set.
- No measurable accuracy difference was observed between the two retrieval methods for this dataset.

### Conclusion
The Haystack pipeline was successfully implemented using PDF conversion, InMemoryDocumentStore, BM25 retrieval, dense retrieval with Sentence Transformers, and an extractive Reader. Both retrieval approaches successfully answered the evaluation questions.

PDF files found: 1
- robotics_reference.pdf


In [1]:
from pathlib import Path

pdf_files = list(Path(
    r"C:\Users\ughan\CynarisInternship\AIML_PROJECT\notebooks"
).glob("*.pdf"))

print("PDF files found:", len(pdf_files))

for pdf in pdf_files:
    print("-", pdf.name)

PDF files found: 1
- robotics_reference.pdf


In [2]:
from pathlib import Path

pdf_files = list(Path(
    r"C:\Users\ughan\CynarisInternship\AIML_PROJECT\notebooks"
).glob("*.pdf"))

print("PDF files found:", len(pdf_files))

for pdf in pdf_files:
    print("-", pdf.name)

PDF files found: 1
- robotics_reference.pdf


In [3]:
from pathlib import Path

project = Path(r"C:\Users\ughan\CynarisInternship\AIML_PROJECT")

pdf_files = list(project.rglob("*.pdf"))

print("PDF files found:", len(pdf_files))

for pdf in pdf_files:
    print("-", pdf)

PDF files found: 27
- C:\Users\ughan\CynarisInternship\AIML_PROJECT\notebooks\robotics_reference.pdf
- C:\Users\ughan\CynarisInternship\AIML_PROJECT\w4d6_venv\Lib\site-packages\matplotlib\mpl-data\images\back.pdf
- C:\Users\ughan\CynarisInternship\AIML_PROJECT\w4d6_venv\Lib\site-packages\matplotlib\mpl-data\images\filesave.pdf
- C:\Users\ughan\CynarisInternship\AIML_PROJECT\w4d6_venv\Lib\site-packages\matplotlib\mpl-data\images\forward.pdf
- C:\Users\ughan\CynarisInternship\AIML_PROJECT\w4d6_venv\Lib\site-packages\matplotlib\mpl-data\images\hand.pdf
- C:\Users\ughan\CynarisInternship\AIML_PROJECT\w4d6_venv\Lib\site-packages\matplotlib\mpl-data\images\help.pdf
- C:\Users\ughan\CynarisInternship\AIML_PROJECT\w4d6_venv\Lib\site-packages\matplotlib\mpl-data\images\home.pdf
- C:\Users\ughan\CynarisInternship\AIML_PROJECT\w4d6_venv\Lib\site-packages\matplotlib\mpl-data\images\matplotlib.pdf
- C:\Users\ughan\CynarisInternship\AIML_PROJECT\w4d6_venv\Lib\site-packages\matplotlib\mpl-data\images

In [4]:
from pathlib import Path

pdf_files = list(Path(
    r"C:\Users\ughan\CynarisInternship\AIML_PROJECT\notebooks"
).glob("*.pdf"))

print("PDF files found:", len(pdf_files))

for pdf in pdf_files:
    print("-", pdf.name)

PDF files found: 5
- computer_vision_robotics.pdf
- pid_control.pdf
- robotics_reference.pdf
- robot_sensors.pdf
- ros2_basics.pdf


In [8]:
from haystack.components.converters import PyPDFToDocument

In [12]:
print("PDF conversion setup ready")

PDF conversion setup ready


In [13]:
from pathlib import Path
from haystack.components.converters import PyPDFToDocument

pdf_folder = Path(
    r"C:\Users\ughan\CynarisInternship\AIML_PROJECT\notebooks"
)

converter = PyPDFToDocument()

all_documents = []

for pdf_file in sorted(pdf_folder.glob("*.pdf")):
    result = converter.run(sources=[str(pdf_file)])
    documents = result["documents"]
    all_documents.extend(documents)
    print(f"{pdf_file.name}: {len(documents)} document(s)")

print("\nTotal documents:", len(all_documents))

computer_vision_robotics.pdf: 1 document(s)
pid_control.pdf: 1 document(s)
robot_sensors.pdf: 1 document(s)
robotics_reference.pdf: 1 document(s)
ros2_basics.pdf: 1 document(s)

Total documents: 5


In [14]:
from haystack.document_stores.in_memory import InMemoryDocumentStore

document_store = InMemoryDocumentStore()

document_store.write_documents(all_documents)

print("Documents indexed:", document_store.count_documents())

Documents indexed: 5


In [15]:
from haystack.components.retrievers.in_memory import InMemoryBM25Retriever

bm25_retriever = InMemoryBM25Retriever(
    document_store=document_store,
    top_k=3
)

print("BM25 retriever ready")

BM25 retriever ready


In [16]:
query = "What does LiDAR measure?"

result = bm25_retriever.run(query=query)

for doc in result["documents"]:
    print("Answer:", doc.content[:500])
    print("-" * 60)

Answer: Robot Sensors Reference
LiDAR
LiDAR uses laser light to measure distances. It is useful for mapping, obstacle detection, and navigation.
IMU
An IMU measures quantities such as linear acceleration and angular velocity. These measurements help
estimate robot motion and orientation.
Wheel Encoders
Wheel encoders measure wheel rotation and can be used to estimate wheel displacement, speed, and
odometry.
Cameras
Cameras provide image data for computer vision tasks such as object detection, tracking, 
------------------------------------------------------------
Answer: Robotics Overview
Robotics combines mechanical engineering, electronics, control systems, and computer science.
Robots use sensors to perceive their surroundings and actuators to produce movement.
Sensors and Perception
LiDAR measures distance using laser light and helps robots build maps. Ultrasonic sensors
estimate distance using sound waves. Cameras provide image data for computer vision. An IMU
measures angular vel

In [17]:
questions = [
    "What does LiDAR measure?",
    "What does an IMU measure?",
    "What do wheel encoders measure?",
    "What is the purpose of sensors in robots?",
    "What is SLAM used for?",
    "What is the purpose of a PID controller?",
    "What is ROS 2 used for?",
    "What do cameras provide for robots?",
    "What do ultrasonic sensors estimate?",
    "What do actuators produce?"
]

for i, question in enumerate(questions, 1):
    result = bm25_retriever.run(query=question)
    print(f"\n{i}. {question}")
    
    for doc in result["documents"][:1]:
        print(doc.content[:300].replace("\n", " "))
        


1. What does LiDAR measure?
Robot Sensors Reference LiDAR LiDAR uses laser light to measure distances. It is useful for mapping, obstacle detection, and navigation. IMU An IMU measures quantities such as linear acceleration and angular velocity. These measurements help estimate robot motion and orientation. Wheel Encoders Whee

2. What does an IMU measure?
Robot Sensors Reference LiDAR LiDAR uses laser light to measure distances. It is useful for mapping, obstacle detection, and navigation. IMU An IMU measures quantities such as linear acceleration and angular velocity. These measurements help estimate robot motion and orientation. Wheel Encoders Whee

3. What do wheel encoders measure?
Robot Sensors Reference LiDAR LiDAR uses laser light to measure distances. It is useful for mapping, obstacle detection, and navigation. IMU An IMU measures quantities such as linear acceleration and angular velocity. These measurements help estimate robot motion and orientation. Wheel Encoders Whee

4

In [18]:
for i, question in enumerate(questions, 1):
    result = bm25_retriever.run(query=question)
    doc = result["documents"][0]

    print(f"\n{i}. {question}")
    print("Source:", doc.meta.get("file_path", "unknown"))
    print("Retrieved:", doc.content[:180].replace("\n", " "))
    


1. What does LiDAR measure?
Source: robot_sensors.pdf
Retrieved: Robot Sensors Reference LiDAR LiDAR uses laser light to measure distances. It is useful for mapping, obstacle detection, and navigation. IMU An IMU measures quantities such as line

2. What does an IMU measure?
Source: robot_sensors.pdf
Retrieved: Robot Sensors Reference LiDAR LiDAR uses laser light to measure distances. It is useful for mapping, obstacle detection, and navigation. IMU An IMU measures quantities such as line

3. What do wheel encoders measure?
Source: robot_sensors.pdf
Retrieved: Robot Sensors Reference LiDAR LiDAR uses laser light to measure distances. It is useful for mapping, obstacle detection, and navigation. IMU An IMU measures quantities such as line

4. What is the purpose of sensors in robots?
Source: pid_control.pdf
Retrieved: PID Control in Robotics PID Purpose A PID controller regulates a system using the error between a desired setpoint and measured output. Proportional Term The proportional

In [19]:
for i, question in enumerate(questions, 1):
    result = bm25_retriever.run(query=question)
    doc = result["documents"][0]

    source = Path(doc.meta.get("file_path", "unknown")).name

    print(f"{i}. {source}")

1. robot_sensors.pdf
2. robot_sensors.pdf
3. robot_sensors.pdf
4. pid_control.pdf
5. robotics_reference.pdf
6. pid_control.pdf
7. ros2_basics.pdf
8. robotics_reference.pdf
9. robot_sensors.pdf
10. robotics_reference.pdf


In [22]:
result = bm25_retriever.run(
    query="What is the purpose of sensors in robots?"
)

for doc in result["documents"]:
    print(doc.content)
    print("=" * 80)

PID Control in Robotics
PID Purpose
A PID controller regulates a system using the error between a desired setpoint and measured output.
Proportional Term
The proportional term produces a response based on the current error.
Integral Term
The integral term accumulates past error and can reduce steady-state error.
Derivative Term
The derivative term responds to the rate of change of error and can reduce overshoot.
Applications
PID control is commonly used for motor speed, robot position, orientation, robotic arms, and balancing robots.

Robotics Overview
Robotics combines mechanical engineering, electronics, control systems, and computer science.
Robots use sensors to perceive their surroundings and actuators to produce movement.
Sensors and Perception
LiDAR measures distance using laser light and helps robots build maps. Ultrasonic sensors
estimate distance using sound waves. Cameras provide image data for computer vision. An IMU
measures angular velocity and linear acceleration, while 

In [26]:
from haystack_integrations.components.readers.transformers import TransformersExtractiveReader

reader = TransformersExtractiveReader(
    top_k=2,
    no_answer=False
)

print("Reader ready")

Reader ready


In [27]:
from haystack import Pipeline

bm25_pipeline = Pipeline()

bm25_pipeline.add_component(
    "retriever",
    bm25_retriever
)

bm25_pipeline.add_component(
    "reader",
    reader
)

bm25_pipeline.connect(
    "retriever.documents",
    "reader.documents"
)

print("BM25 → Reader pipeline ready")

BM25 → Reader pipeline ready


In [28]:
result = bm25_pipeline.run(
    {
        "retriever": {
            "query": "What does LiDAR measure?"
        },
        "reader": {
            "query": "What does LiDAR measure?"
        }
    }
)

for answer in result["reader"]["answers"]:
    print("Answer:", answer.data)
    print("Score:", answer.score)

config.json:   0%|          | 0.00/729 [00:00<?, ?B/s]

c:\Users\ughan\CynarisInternship\AIML_PROJECT\venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ughan\.cache\huggingface\hub\models--deepset--roberta-base-squad2-distilled. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
c:\Users\ughan\CynarisInternship\AIML_PROJECT\venv\Lib\site-packages\huggingface

model.safetensors: reconstructing file:   0%|          |  0.00B /  496MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

...

vocab.json:   0%|          | 0.00/798k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Answer: distance
Score: 0.8347734212875366
Answer: distances
Score: 0.8244938254356384


In [29]:
bm25_reader_results = []

for i, question in enumerate(questions, 1):
    result = bm25_pipeline.run(
        {
            "retriever": {"query": question},
            "reader": {"query": question}
        }
    )

    answers = result["reader"]["answers"]

    best_answer = answers[0].data if answers else "No answer"

    bm25_reader_results.append(best_answer)

    print(f"{i}. {question}")
    print(f"   Answer: {best_answer}")

1. What does LiDAR measure?
   Answer: distance
2. What does an IMU measure?
   Answer: angular velocity and linear acceleration
3. What do wheel encoders measure?
   Answer: wheel rotation
4. What is the purpose of sensors in robots?
   Answer: to perceive their surroundings
5. What is SLAM used for?
   Answer: allows a robot to build a map while estimating its own position
6. What is the purpose of a PID controller?
   Answer: to regulate a system
7. What is ROS 2 used for?
   Answer: to build distributed robot applications
8. What do cameras provide for robots?
   Answer: image data for computer vision
9. What do ultrasonic sensors estimate?
   Answer: distance
10. What do actuators produce?
   Answer: movement


In [34]:
from haystack.components.retrievers.in_memory import InMemoryEmbeddingRetriever

print("Dense retriever import ready")

Dense retriever import ready


In [35]:
dense_retriever = InMemoryEmbeddingRetriever(
    document_store=document_store,
    top_k=3
)

print("Dense retriever ready")

Dense retriever ready


In [36]:
from haystack_integrations.components.embedders.sentence_transformers import (
    SentenceTransformersDocumentEmbedder
)
from haystack.document_stores.types import DuplicatePolicy

document_embedder = SentenceTransformersDocumentEmbedder(
    model="sentence-transformers/all-MiniLM-L6-v2"
)

document_embedder.warm_up()

embedded_result = document_embedder.run(documents=all_documents)

document_store.write_documents(
    embedded_result["documents"],
    policy=DuplicatePolicy.OVERWRITE
)

print("Document embeddings created and indexed:", document_store.count_documents())

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Document embeddings created and indexed: 5


In [37]:
from haystack_integrations.components.embedders.sentence_transformers import (
    SentenceTransformersTextEmbedder
)

text_embedder = SentenceTransformersTextEmbedder(
    model="sentence-transformers/all-MiniLM-L6-v2"
)

text_embedder.warm_up()

print("Query embedder ready")

Query embedder ready


In [39]:
from haystack_integrations.components.readers.transformers import TransformersExtractiveReader

dense_reader = TransformersExtractiveReader(
    top_k=2,
    no_answer=False
)

print("Dense reader ready")

Dense reader ready


In [41]:
from haystack_integrations.components.embedders.sentence_transformers import (
    SentenceTransformersTextEmbedder
)

dense_text_embedder = SentenceTransformersTextEmbedder(
    model="sentence-transformers/all-MiniLM-L6-v2"
)

dense_text_embedder.warm_up()

print("Dense text embedder ready")

Dense text embedder ready


In [43]:
from haystack.components.retrievers.in_memory import InMemoryEmbeddingRetriever

dense_retriever_v2 = InMemoryEmbeddingRetriever(
    document_store=document_store,
    top_k=3
)

print("Fresh Dense retriever ready")

Fresh Dense retriever ready


In [45]:
from haystack.components.retrievers.in_memory import InMemoryEmbeddingRetriever
from haystack_integrations.components.embedders.sentence_transformers import (
    SentenceTransformersTextEmbedder
)
from haystack_integrations.components.readers.transformers import (
    TransformersExtractiveReader
)

fresh_text_embedder = SentenceTransformersTextEmbedder(
    model="sentence-transformers/all-MiniLM-L6-v2"
)
fresh_text_embedder.warm_up()

fresh_dense_retriever = InMemoryEmbeddingRetriever(
    document_store=document_store,
    top_k=3
)

fresh_dense_reader = TransformersExtractiveReader(
    top_k=2,
    no_answer=False
)

print("Fresh Dense components ready")

Fresh Dense components ready


In [46]:
dense_pipeline = Pipeline()

dense_pipeline.add_component("text_embedder", fresh_text_embedder)
dense_pipeline.add_component("retriever", fresh_dense_retriever)
dense_pipeline.add_component("reader", fresh_dense_reader)

dense_pipeline.connect(
    "text_embedder.embedding",
    "retriever.query_embedding"
)

dense_pipeline.connect(
    "retriever.documents",
    "reader.documents"
)

print("Dense → Reader pipeline ready")

Dense → Reader pipeline ready


In [47]:
q = "What does LiDAR measure?"

dense_result = dense_pipeline.run({
    "text_embedder": {
        "text": q
    },
    "reader": {
        "query": q
    }
})

print("Question:", q)

for answer in dense_result["reader"]["answers"]:
    print("Answer:", answer.data)
    print("Score:", answer.score)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Question: What does LiDAR measure?
Answer: distance
Score: 0.8347734212875366
Answer: distances
Score: 0.8244938254356384


In [48]:
questions = [
    "What does LiDAR measure?",
    "What does an IMU measure?",
    "What do wheel encoders measure?",
    "What is the purpose of sensors in robots?",
    "What is SLAM used for?",
    "What is the purpose of a PID controller?",
    "What is ROS 2 used for?",
    "What do cameras provide for robots?",
    "What do ultrasonic sensors estimate?",
    "What do actuators produce?"
]

dense_answers = []

for i, q in enumerate(questions, 1):
    result = dense_pipeline.run({
        "text_embedder": {"text": q},
        "reader": {"query": q}
    })

    answers = result["reader"]["answers"]

    if answers:
        answer = answers[0].data
    else:
        answer = "No answer"

    dense_answers.append(answer)

    print(f"{i}. {q}")
    print(f"   Answer: {answer}")
    print()

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

1. What does LiDAR measure?
   Answer: distance



Batches:   0%|          | 0/1 [00:00<?, ?it/s]

2. What does an IMU measure?
   Answer: angular velocity and linear acceleration



Batches:   0%|          | 0/1 [00:00<?, ?it/s]

...

8. What do cameras provide for robots?
   Answer: image data for computer vision



Batches:   0%|          | 0/1 [00:00<?, ?it/s]

9. What do ultrasonic sensors estimate?
   Answer: distance



Batches:   0%|          | 0/1 [00:00<?, ?it/s]

10. What do actuators produce?
   Answer: movement



## Evaluation and Comparison

The Haystack RAG pipeline was evaluated using 10 questions across 5 indexed PDF documents.

### BM25 Retriever + Reader

- Questions tested: 10
- Correct answers: 10/10
- Accuracy: 100%

### Dense Retriever + Reader

- Questions tested: 10
- Correct answers: 10/10
- Accuracy: 100%
- Embedding model: `sentence-transformers/all-MiniLM-L6-v2`

### Comparison

| Method | Questions | Correct | Accuracy |
|---|---:|---:|---:|
| BM25 + Reader | 10 | 10/10 | 100% |
| Dense + Reader | 10 | 10/10 | 100% |

Both retrieval methods produced correct answers for all 10 questions. BM25 performed well for keyword-based retrieval, while the Dense Retriever used semantic embeddings to retrieve relevant documents.

### Conclusion

Both BM25 and Dense retrieval achieved 100% accuracy on the evaluated question set. The Dense Retriever provides semantic retrieval using embeddings, while BM25 relies on lexical keyword matching. For this small evaluation dataset, both approaches performed equally well.